# Attention Benchmark — Visualizations

Loads results CSVs and produces:
- **Heatmap:** all 33 models × 3 tasks, colored by pass rate, sorted by mean score
- **Score distributions:** per-task bar charts showing how many models landed at each distinct score level

Both plots are saved as PNG files.

> **Path note:** `RESULTS_DIR` below uses a relative path for local use. If running in Kaggle with CSVs uploaded as a dataset, update `RESULTS_DIR` to the appropriate `/kaggle/input/...` path.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RESULTS_DIR = "../results"  # update to /kaggle/input/<dataset-name> if running in Kaggle

In [ ]:
selective = pd.read_csv(f"{RESULTS_DIR}/selective_attention_results.csv")
sustained = pd.read_csv(f"{RESULTS_DIR}/sustained_attention_results.csv")
divided   = pd.read_csv(f"{RESULTS_DIR}/divided_attention_results.csv")

def clean_name(raw):
    """Strip provider prefix, @version suffix, and -YYYY-MM-DD date suffix."""
    name = raw.split("/", 1)[-1]
    name = re.sub(r"@.*$", "", name)
    name = re.sub(r"-\d{4}-\d{2}-\d{2}$", "", name)
    return name

for df in (selective, sustained, divided):
    df["model"] = df["model"].apply(clean_name)

print(f"Models loaded — selective: {len(selective)}, sustained: {len(sustained)}, divided: {len(divided)}")

## Heatmap — All Models × All Tasks

In [ ]:
merged = (
    selective[["model", "pass_rate"]].rename(columns={"pass_rate": "Selective"})
    .merge(sustained[["model", "pass_rate"]].rename(columns={"pass_rate": "Sustained"}), on="model")
    .merge(divided[["model", "pass_rate"]].rename(columns={"pass_rate": "Divided"}), on="model")
)
merged["mean"] = merged[["Selective", "Sustained", "Divided"]].mean(axis=1)
merged = merged.sort_values("mean", ascending=False).reset_index(drop=True)

pivot = merged.set_index("model")[["Selective", "Sustained", "Divided"]]
annot = pivot.apply(lambda col: col.apply(lambda x: f"{x:.0%}"))

fig, ax = plt.subplots(figsize=(7, 14))
sns.heatmap(
    pivot,
    annot=annot,
    fmt="",
    cmap="RdYlGn",
    vmin=0.0,
    vmax=1.0,
    linewidths=0.5,
    linecolor="#d0d0d0",
    cbar_kws={"label": "Pass Rate", "shrink": 0.35, "pad": 0.02},
    annot_kws={"size": 8},
    ax=ax,
)
ax.set_title("Attention Benchmark\nAll Models × All Tasks", fontsize=13, pad=14)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="y", labelsize=7.5)
ax.tick_params(axis="x", labelsize=10, bottom=False, top=True, labelbottom=False, labeltop=True)

plt.tight_layout()
plt.savefig("attention_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: attention_heatmap.png")

## Score Distributions — Per Task

In [ ]:
cmap = plt.cm.RdYlGn

task_specs = [
    ("Selective Attention", selective),
    ("Sustained Attention", sustained),
    ("Divided Attention",   divided),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, (title, df) in zip(axes, task_specs):
    counts = df["pass_rate"].value_counts().sort_index(ascending=False)
    x_labels = [f"{v:.0%}" for v in counts.index]
    colors = [cmap(v) for v in counts.index]

    bars = ax.bar(
        x_labels,
        counts.values,
        color=colors,
        edgecolor="white",
        linewidth=0.8,
        width=0.6,
    )
    for bar, count in zip(bars, counts.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.15,
            str(count),
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
        )

    ax.set_title(title, fontsize=11, pad=8)
    ax.set_xlabel("Pass Rate", fontsize=9)
    ax.set_ylabel("Models" if ax is axes[0] else "", fontsize=9)
    ax.set_ylim(0, counts.max() + 3)
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.tick_params(axis="y", labelsize=8)
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Score Distribution by Task  (n = 33 models)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("attention_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: attention_distributions.png")